In [1]:
# Core
import os
import time
import requests
import pandas as pd
import numpy as np

# Market data
import yfinance as yf

# NLP
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Statistics
import statsmodels.api as sm


c:\Users\Meet\Desktop\Desktop Folders\Work Projects\Dissertation UEL\my-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# YahooFinance

def fetch_ohlcv(symbol: str, period: str = "1y") -> pd.DataFrame:
    data = yf.Ticker(symbol)
    history = data.history(period=period)
    
    history.drop(['Dividends', 'Stock Splits'], axis=1, inplace=True)
    history.index = history.index.tz_localize(None)
    
    return history


In [13]:
def fetch_news(symbols, start, end, api_key, api_secret):
    url = "https://data.alpaca.markets/v1beta1/news?"
    headers = {
        "accept": "application/json",
        "APCA-API-KEY-ID": api_key,
        "APCA-API-SECRET-KEY": api_secret
    }
    
    all_articles = []
    page_token = None

    while True:
        params = {
            "start": start,
            "end": end,
            "symbols": ",".join(symbols),
            "limit": 50,
            "include_content": True,
            "exclude_contentless": True,
            "sort": "asc",
            "page_token": page_token
        }

        r = requests.get(url, headers=headers, params=params)
        r.raise_for_status()
        data = r.json()

        all_articles.extend(data["news"])
        page_token = data.get("next_page_token")

        if page_token is None:
            break

        time.sleep(0.2)  # API safety

    return pd.DataFrame(all_articles)


In [4]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = text.replace("\n", " ")
    return text


In [5]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"


In [6]:

#tokenizer integration for pre training data
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
# model integration to infer it with pre trained data
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
# model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

labels = {0: "negative", 1: "neutral", 2: "positive"}


c:\Users\Meet\Desktop\Desktop Folders\Work Projects\Dissertation UEL\my-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
def score_sentiment(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    scores = torch.softmax(outputs.logits, dim=1).numpy()
    sentiment = scores[:, 2] - scores[:, 0]  # positive − negative

    return sentiment


In [8]:
def aggregate_daily_sentiment(news_df):
    news_df["created_at"] = pd.to_datetime(news_df["created_at"]).dt.date
    news_df["content"] = news_df["content"].apply(clean_text)

    news_df["sentiment"] = score_sentiment(news_df["content"].tolist())

    daily_sentiment = (
        news_df
        .groupby("created_at")["sentiment"]
        .mean()
        .to_frame("daily_sentiment")
    )

    return daily_sentiment


In [9]:
def compute_returns(price_df):
    returns = price_df["Close"].pct_change().dropna()
    returns.name = "returns"
    return returns


In [10]:
def build_research_table(price_df, sentiment_df):
    df = price_df.copy()
    df["date"] = df.index.date

    merged = df.merge(
        sentiment_df,
        left_on="date",
        right_index=True,
        how="inner"
    )

    merged["returns"] = merged["Close"].pct_change()
    merged.dropna(inplace=True)

    return merged


In [11]:
def run_regression(df):
    X = sm.add_constant(df["daily_sentiment"])
    y = df["returns"]

    model = sm.OLS(y, X).fit()
    return model


In [15]:
symbols = ["MSFT"]

price = fetch_ohlcv("MSFT", "1y")

news = fetch_news(
    symbols=symbols,
    start="2024-01-03T00:00:00Z",
    end="2025-10-03T00:00:00Z",
    api_key="PKDOOOL44COZGZB6E7KNQAFMOF",
    api_secret="FZSHhs41VmSSU8z7otcQfHMJEFFuqHzSuMLZVuixrexZ"
)

daily_sentiment = aggregate_daily_sentiment(news)

dataset = build_research_table(price, daily_sentiment)

results = run_regression(dataset)

print(results.summary())


RuntimeError: [enforce fail at alloc_cpu.cpp:121] data. DefaultCPUAllocator: not enough memory: you tried to allocate 6874988544 bytes.